# Snake played by a CNN and a mixture of expert controllers

*ai · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/ai/snake_moe_control)

A game-playing agent built from two parts: a small CNN that looks at the board and a control network that picks the move, here a mixture of experts whose router sends each game to one of $K$ small expert networks. The game is Snake on a $10 \times 10$ grid, written entirely as tensors so that a batch of games advances in one step. The controllers learn by imitating a planning teacher (breadth-first search and flood fill, also tensor operations). Four controllers with the same weight budget are compared: a dense mixture, a Switch top-1 router, a dense-trained router used top-1, and one plain MLP; then the routing is measured against the game state, to see what, if anything, each expert is used for. All four eat about 20 pieces of food per game where a random policy eats 0.2, two thirds of the teacher's score; at this size the plain MLP does as well as any mixture.

**Vector form**

$$z = \sum_k G_k(\mathbf h)\, f_k(\mathbf h),\qquad \mathbf h = \mathrm{CNN}(\text{view}),\qquad p = \mathrm{softmax}(W^G \mathbf h + b^G),\qquad G = p\ \text{(dense)}\ \text{or}\ \mathrm{onehot}(\arg\max p) \odot p\ \text{(top-1)},\qquad \mathcal L = \mathrm{CE}(z, a^{\ast}) + \alpha\, K \sum_k f_k P_k$$

**Index (tensor) form**

$$D_{mbks} = G_{mbk}\,[s = \mathrm{rank}_k(b)],\qquad U_{mksf} = D_{mbks}\, h_{mbf},\qquad Y_{mksa} = f_k(U_{mks})_a,\qquad z_{mba} = D_{mbks}\, g_{mbk}\, Y_{mksa},\qquad \mathrm{age}^{\prime}_{brc} = \max(\mathrm{age}_{brc} - [\neg \mathrm{eat}_b], 0),\ \mathrm{age}^{\prime}_{b,\mathbf x^{\prime}_b} = \ell^{\prime}_b$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\mathrm{age}_{brc}$ | `State.age` | (B, N, N) | the board of game $b$: 0 = empty, otherwise the moves until that body cell is free again (head = length $\ell_b$, tail = 1) |
| $\mathbf x'_b = \mathbf x_b + \Delta_{d'_b}$ | `step(s, action, u, hunger_limit)` | (B, 2) | all $B$ games move at once; $d'_b = (d_b + a_b - 1) \bmod 4$ for the relative action $a_b$ (left, straight, right); a wall or a cell with age $> 1$ ends the game (the tail cell, age 1, is leaving) |
| $\mathrm{food}'_b$ | `place_food(age, u)` | (B, 2) | after eating, the empty cell with the largest of the seeded uniform numbers $u_{brc}$: never on the body |
| $v_{bcij}$ | `planes(s)` | (B, 4, 2N-1, 2N-1) | what the CNN sees: wall, body, age/length and food, centred on the head with the heading pointing up, so the three moves are always the same three neighbours |
| $h_{mbf}$ | `cnn_features(x, w)` | (M, B, F) | CNN features; the model axis $m$ (variant × seed) becomes the groups of one grouped convolution per layer |
| $p_{mbk}$ | `router_probs(h, router)` | (M, B, K) | router: softmax over the $K$ experts, rows sum to 1 |
| $D_{mbks}$ | `dispatch(G, C)` | (M, B, K, C) | game $b$ to slot $s$ of expert $k$ (its rank among the games routed there), imported from moe_dag_pipeline |
| $Y_{mksa}$ | `einsum('mksf,mkfh->mksh', U, W1) …` | (M, K, C, 3) | all experts in one batched product, each on its own buffer of games: three action logits |
| $z_{mba}$ | `einsum('mbks,mbk,mksa->mba', D, g, Y)` | (M, B, 3) | combine; dense: $z = p_{mbk} y_{mkba}$ with every expert on every game |
| $\mathcal L_{\mathrm{aux}}$ | `load_balance_loss(p, top1(p))` | (M,) | Switch load balancing, $K \sum_k f_k P_k$: 1 when balanced, $K$ when collapsed |
| $a^{\ast}_b$ | `teacher_actions(s)` | (B,) | the teacher: no deadly move, then a move whose free region (flood fill) holds the snake, then the shortest path to the food (breadth-first search as repeated tensor relaxation) |

Idea. Jacky Ka Long Ko suggested a game-playing agent made of a CNN for perception and a control network with mixture-of-experts routing, and asked for the tensor routing part above all. This problem builds a small one on Snake, a generic grid game written from scratch here (own pixel look, no assets). Game. $B$ games on an $N \times N$ board ($N = 10$) are one set of tensors. The board stores each body cell's age (moves until it is free again: the head has age = length, the tail 1), so one move of every game is a handful of tensor operations: the head steps to $\mathbf x + \Delta_{d'}$ with the new heading $d' = (d + a - 1) \bmod 4$ for the relative action $a$ (left, straight, right); a wall or a cell of age $> 1$ ends the game (the tail cell is leaving, so it is free); without food every age drops by 1, with food the snake keeps its tail and grows by one; the new food is the empty cell with the largest of $N^2$ seeded uniform numbers, so it never lands on the body. A game also ends after $N^2$ moves without food or when the snake fills the board. Tests: wall and self collision, the free tail cell, growth by one, food never on the body, the ages always exactly $\ell, \ell - 1, \dots, 1$, and the batched step equal to a plain per-game Python loop given the same random numbers. Agent. Perception is a CNN (3×3 convolution with 8 channels, a stride-2 3×3 convolution with 16, a linear layer to $F = 64$ features; 84 472 weights) on a view centred on the head with the heading pointing up, four planes: wall, body, age/length (which cells free up first) and food. In this frame the three relative moves are always the same three neighbours of the centre (a test checks that a move is deadly exactly when that neighbour is a wall or a body cell that stays). Control is a mixture of $K = 4$ experts, each an MLP $64 \to 32 \to 3$ giving action logits, behind a softmax router $p = \mathrm{softmax}(W^G \mathbf h + b^G)$: 8976 weights. The routed layer is the one of moe_dag_pipeline, imported from it: a dispatch tensor puts each game into its expert's buffer (cumulative ranks, as in GShard and Switch), all experts run in one batched einsum, and a combine einsum brings the logits back. Exact checks: with a fixed one-hot router the layer equals running each game's expert alone (to $10^{-6}$), a uniform dense router gives the mean of the experts, router rows sum to 1, and the load-balancing loss on known cases (1 when balanced, $K$ when collapsed, 1.1 on a hand-computed example). Four controllers, same budget: (a) a dense mixture $z = \sum_k p_k f_k$ (every expert on every game); (b) Switch top-1, only the most probable expert runs and its logits are scaled by $p_k$; (c) the dense-trained model of (a) run with top-1 routing, no further training; (d) one MLP $64 \to 132 \to 3$ (8979 weights, the same as the mixture). (a) and (b) both add the Switch load-balancing loss with $\alpha = 0.01$. Each variant has two seeds; seed $s$ starts from the same CNN and expert weights in every variant, and all six trained models share one model axis (the CNN's grouped convolution), each with its own minibatches. Teacher and training. The teacher is a planner written with the same tensors: breadth-first search as repeated relaxation over the free cells ($\mathrm{dist} \leftarrow \min(\mathrm{dist}, 1 + \min_{\mathrm{nbr}} \mathrm{dist})$) and, in the same loop, a flood fill that labels each free region by its smallest cell index. It never takes a deadly move, prefers a move whose free region can hold the snake, then the shortest path to the food, then the larger region; its distance field equals a textbook queue-based BFS in a test. Training is behaviour cloning (Pomerleau 1989): cross-entropy on the teacher's moves, from 54 040 states of 192 teacher games, 600 AdamW steps of 128 states per model, with one DAgger round (Ross et al. 2011) after 360 steps: the students play 48 games each, the teacher labels the 38 267 states they reach, and training continues on both. No reinforcement-learning fine-tune is done; deep Q-learning (Mnih et al. 2015) is the classic way to learn such games from reward alone, which would take far longer on a CPU. Results (default run, 128 evaluation games shared by every policy: same starts and the same food stream, at most 300 moves). Food eaten per game, mean over games, with the two seeds: random 0.16 (dies after 15 moves); (a) dense 20.7 (20.9, 20.4); (b) Switch 20.4 (21.1, 19.6); (c) dense→top-1 18.1 (18.4, 17.8); (d) monolithic 21.7 (21.8, 21.6); teacher 31.0 (287 of 300 moves survived on average, against 147–179 for the students). The students agree with the teacher on 95–96 % of training states, yet score a third less: small errors compound over a game, the usual weakness of behaviour cloning. A slow test checks that a trained policy beats random by a clear margin. Routing, shown even where it is not a specialisation. The dense routers use all four experts (loads 0.16–0.38). What they track best is the move itself: the mutual information between the routed expert and the controller's move is 0.51 and 0.56 bit (2 bit is the most possible with four experts); in seed 1 one expert takes almost every left and right turn and almost every state with no free cell ahead, while the other three share the straight moves. Free cells ahead carry 0.20–0.28 bit, the distance to the food 0.08–0.12 and the body length 0.02–0.13 (a drift from short to long snakes in seed 1 only). So the experts split by what to do (turn or keep going), not by a strategy such as food-chasing versus tail avoidance; since an expert's output decides the move, this division is partly built in. The Switch routers collapsed despite the balancing loss: seed 1 sends 85 % of the states to one expert and 15 % to a second, seed 2 73 % and 26 %, two experts unused in each; the symmetry-breaking difficulty seen in moe_dag_pipeline appears here as collapse. It costs nothing in score, because one expert with the shared CNN (which holds most of the weights) already plays as well as four. Running the dense-trained router top-1 (c) keeps the dense routing (0.60 and 0.67 bit with the move) but loses 2.6 food per game: the dense model blends its experts and is not quite separable. What holds and what does not. Exact: the batched game equals the reference loop, the routed layer with a fixed router equals per-game expert calls, and the dense mixture with a uniform router is the expert mean. Measured on one board size with two seeds: imitation brings every controller to about two thirds of the teacher, and no mixture beats the plain MLP with the same weights (the MLP is ahead by about 1 food per game in both seeds, a small gap at two seeds). Mixtures of experts pay off when many experts hold far more weights than one input can use; a four-expert controller behind a CNN that does most of the work demonstrates the routing, not its advantage. Related work. Adaptive mixtures of local experts (Jacobs, Jordan, Nowlan and Hinton 1991) introduced a gate that assigns each case to one of several competing experts; the sparsely gated MoE layer (Shazeer et al. 2017) and the Switch Transformer (Fedus et al. 2021, top-1 routing and the load-balancing loss used here) scaled it up. Deep Q-networks (Mnih et al. 2015) learned Atari games from pixels with a CNN and reward alone; ALVINN (Pomerleau 1989) steered a vehicle by imitating a human driver, the origin of behaviour cloning, and DAgger (Ross, Gordon and Bagnell 2011) fixes its compounding errors by labelling the states the learner itself visits. Device and limits. CPU by default: about 80 s of teacher, training and evaluation on a busy 6-core machine (6.6 s teacher games, 6.0 s DAgger, 53 s training, 16 s evaluation) plus about 30 s for the GIF; a GPU (pixi run -e cuda) speeds up the CNN but not the teacher's many small searches. The GIF shows (c), whose router uses all experts and runs exactly one per game; routing.png shows all three routed variants, routing.csv and scores.csv hold the numbers.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/ai/snake_moe_control/main.py
"""Main loop: B games of Snake stepped as tensors; a CNN sees each board, a mixture-of-experts controller picks the
move. Imitation of a planning teacher (+ one DAgger round) trains four controllers at once on one model axis:
(a) dense mixture, (b) Switch top-1, (c) dense-trained then top-1, (d) a monolithic MLP with as many weights."""

import math
import time
from pathlib import Path

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve
from . import tensors as T


def run(params: dict, out_dir: Path) -> Path:
    fast = fast_mode()
    out_dir = Path(out_dir)
    cfg = T.Config(
        grid=params["grid"],
        seeds=params["seeds"],
        experts=params["experts"],
        steps=40 if fast else params["steps"],
        balance_weight=params["balance_weight"],
        dagger=bool(params["dagger"]),
        teacher_games=32 if fast else 192,
        teacher_steps=40 if fast else 300,
        dagger_games=8 if fast else 48,
        dagger_steps=20 if fast else 200,
        eval_games=16 if fast else params["eval_games"],
        eval_steps=40 if fast else 300,
        show_steps=40 if fast else 160,
    )
    dev = device()
    t0 = time.time()
    res = solve.run_all(cfg, dev)
    S, K = cfg.seeds, cfg.experts
    print(f"{cfg.grid}×{cfg.grid} board, {S} seeds per variant on one model axis ({3 * S} trained models), "
          f"{dev.type.upper()}: {time.time() - t0:.0f} s")  # fmt: skip
    print("    " + ", ".join(f"{k} {v:.1f} s" for k, v in res["time"].items()))
    n_teacher, n_all = res["pool"]
    print(f"imitation data: {n_teacher} teacher states" + (f" + {n_all - n_teacher} DAgger states (students' "
          "games, teacher's labels)" if cfg.dagger else ""))  # fmt: skip
    pr = res["params"]
    print(f"weights per model: CNN {pr['cnn']}, MoE controller (router + {K} experts) {pr['moe']}, monolithic MLP "
          f"{pr['mono']}")  # fmt: skip
    print(f"score on {cfg.eval_games} shared games (≤ {cfg.eval_steps} moves, ends on a wall, the body, or "
          f"{cfg.hunger_limit()} moves without food):")  # fmt: skip
    print("    policy               food/game   per seed        moves survived   agreement with teacher (train)")
    acc = res["acc"][-50:].mean(0)
    for v in plot.POLICIES:
        x = res["food"][v]
        seeds = " ".join(f"{float(s):5.2f}" for s in x.mean(-1)) if x.dim() > 1 else ""
        ag = ""
        if v in ("dense", "switch", "mono"):
            j = ("dense", "switch", "mono").index(v)
            ag = f"{float(acc[j * S : (j + 1) * S].mean()):.3f}"
        print(f"    {T.LABELS[v]:20s} {float(x.mean()):8.2f}   {seeds:15s} {float(res['steps'][v].mean()):8.0f}"
              f"         {ag}")  # fmt: skip
    sp = res["spec"]
    print("routing (routed models, seed by seed): expert loads; mutual information (bit) between the routed expert "
          f"and the state (max log2 {K} = {math.log2(K):.2f})")  # fmt: skip
    for r, v in enumerate(("dense", "switch", "dense_top1")):
        for s in range(S):
            m = r * S + s
            loads = " ".join(f"{float(x):.2f}" for x in sp["load"][m])
            mis = ", ".join(f"{k} {float(sp['mi'][k][m]):.2f}" for k in ("dist", "ahead", "length", "turn"))
            print(f"    {T.LABELS[v]:18s} seed {s + 1}: loads {loads}; MI: {mis}")
    print("    (dist = distance to food, ahead = free cells straight ahead, length = body length, turn = the move "
          "the controller makes)")  # fmt: skip

    plot.write_tables(res, out_dir)
    plot.routing_png(res, out_dir / "routing.png")
    return plot.animate(res, out_dir / "snake_moe_control.gif", n_frames=6 if fast else 50)


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/ai/snake_moe_control/equation.py
r"""The laws as tensor operations: one step of B Snake games, the network's view, the teacher's breadth-first search,
the CNN, and the mixture-of-experts controller (dispatch → experts → combine, the layer of moe_dag_pipeline).

Indices: m = model (variant × seed), b = game, r, c = board cell, i, j = view cell, k = expert, s = slot in an
expert's buffer, f = CNN feature, h = hidden unit, a = action. Repeated indices are summed unless in the output.

    move           d'_b = (d_b + a_b − 1) mod 4,   x'_b = x_b + Δ_{d'_b}            (relative action a ∈ {0, 1, 2})
    collision      dead_b = [x'_b outside the board] ∨ [age_{b, x'_b} > 1]    (age 1 = the tail, which moves away)
    body           age'_{brc} = max(age_{brc} − 1, 0) (no food) or age_{brc} (food: the tail stays),
                   age'_{b, x'_b} = length'_b,   length'_b = length_b + [x'_b = food_b]
    food respawn   food'_b = argmax_{rc} u_{brc} [age'_{brc} = 0]          (u uniform from the seeded generator)
    view           v_{bij} = board_b(x_b + (R − i) Δ_{d_b} + (j − R) Δ_{d_b + 1}),  R = N − 1  (head-centred)
    distance       dist ← min(dist, 1 + min_{neighbours} dist) on free cells until fixed       (BFS by relaxation)
    flood fill     label ← min(label, min_{neighbours} label): one label per free region, area = its cell count
    CNN            h_{mbf} = relu(W^{fc}_{m·f} · relu(conv_2(relu(conv_1(v_{mb})))) + b_{mf})
    router         p_{mbk} = softmax_k(h_{mbf} W^G_{mfk} + b^G_{mk})
    expert         y_{mkba} = W2_{mkha} relu(W1_{mkfh} h_{mbf} + b1_{mkh}) + b2_{mka}
    dense          z_{mba} = p_{mbk} y_{mkba}
    top-1 (Switch) G_{mbk} = onehot(argmax_k p_{mbk}),  D_{mbks} = G_{mbk} [s = rank of b in group k],
                   U_{mksf} = D_{mbks} h_{mbf},  Y_{mksa} = f_k(U_{mks}),  z_{mba} = D_{mbks} G_{mbk} p_{mbk} Y_{mksa}
    load balance   ℒ_aux = K Σ_k f_k P_k,  f_k = ⟨G_{bk}⟩_b,  P_k = ⟨p_{bk}⟩_b                 (Fedus et al. 2021)
    imitation      ℒ = −⟨log softmax_a z_{mba}[a*_b]⟩_b   (a* = the teacher's action; behaviour cloning)

dispatch, combine, top1 and load_balance_loss are imported from problems/ai/moe_dag_pipeline.
"""

import torch
import torch.nn.functional as F

from problems.ai.moe_dag_pipeline.equation import combine, cross_entropy, dispatch, load_balance_loss, top1

from . import tensors as T

__all__ = ["combine", "cross_entropy", "dispatch", "load_balance_loss", "top1"]  # re-exported for the tests
FAR = 10**6  # distance of an unreachable cell


# ---------------------------------------------------------------- the game
def _cell(age: torch.Tensor, rc: torch.Tensor) -> torch.Tensor:
    """age (B, N, N) at cells rc (B, ..., 2) (clamped inside the board) → (B, ...)."""
    n = age.shape[1]
    rc = rc.clamp(0, n - 1)
    idx = (rc[..., 0] * n + rc[..., 1]).reshape(len(rc), -1)
    return age.flatten(1).gather(1, idx).view(rc.shape[:-1])


def _inside(rc: torch.Tensor, n: int) -> torch.Tensor:
    return ((rc >= 0) & (rc < n)).all(-1)


def place_food(age: torch.Tensor, u: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """A uniformly random empty cell per game: argmax of u over the empty cells → food (B, 2), any_empty (B,)."""
    n = age.shape[1]
    empty = age.flatten(1) == 0
    k = torch.where(empty, u, torch.full_like(u, -1.0)).argmax(1)
    return torch.stack([k // n, k % n], 1), empty.any(1)


def reset(B: int, n: int, gen: torch.Generator) -> T.State:
    """B new games: a 3-long snake with its head in the board's interior, a random heading, the body behind it."""
    head = torch.randint(2, n - 2, (B, 2), generator=gen)
    d = torch.randint(4, (B,), generator=gen)
    k = torch.arange(T.START_LENGTH)
    body = head[:, None] - k[None, :, None] * T.DIRS[d][:, None]  # (B, 3, 2) head, neck, tail
    age = torch.zeros(B, n * n, dtype=torch.long)
    age.scatter_(1, body[..., 0] * n + body[..., 1], (T.START_LENGTH - k).expand(B, -1).contiguous())
    age = age.view(B, n, n)
    food, _ = place_food(age, torch.rand(B, n * n, generator=gen))
    z = torch.zeros(B, dtype=torch.long)
    return T.State(age=age, head=head, dir=d, food=food, length=torch.full((B,), T.START_LENGTH),
                   alive=torch.ones(B, dtype=torch.bool), hunger=z, eaten=z.clone(), steps=z.clone())  # fmt: skip


def step(s: T.State, action: torch.Tensor, u: torch.Tensor, hunger_limit: int) -> T.State:
    """All B games one move: action (B,) relative, u (B, N²) uniform numbers for the food respawn. Games that have
    ended stay frozen. A game ends on a wall, on its own body, after hunger_limit moves without food, or when the
    snake fills the board."""
    B, n, _ = s.age.shape
    d = (s.dir + action - 1) % 4
    new = s.head + T.DIRS.to(s.head.device)[d]  # (B, 2)
    inside = _inside(new, n)
    hit = ~inside | (_cell(s.age, new) > 1)  # the tail (age 1) leaves this step, so its cell is free
    eat = inside & (new == s.food).all(-1)
    hunger = torch.where(eat, 0, s.hunger + 1)
    dies = s.alive & (hit | (hunger >= hunger_limit))
    moves = s.alive & ~dies
    grow = moves & eat
    length = s.length + grow
    age = torch.where((moves & ~eat)[:, None, None], (s.age - 1).clamp(min=0), s.age)
    nc = new.clamp(0, n - 1)
    flat = age.flatten(1).clone()
    idx = (nc[:, 0] * n + nc[:, 1])[:, None]
    flat.scatter_(1, idx, torch.where(moves, length, flat.gather(1, idx)[:, 0])[:, None])
    age = flat.view(B, n, n)
    food_new, room = place_food(age, u)
    m2 = moves[:, None]
    return T.State(
        age=age,
        head=torch.where(m2, nc, s.head),
        dir=torch.where(moves, d, s.dir),
        food=torch.where((grow & room)[:, None], food_new, s.food),
        length=length,
        alive=moves & ~(grow & ~room),  # a full board ends the game (a win)
        hunger=torch.where(moves, hunger, s.hunger),
        eaten=s.eaten + grow,
        steps=s.steps + moves,
    )


# ---------------------------------------------------------------- what the network sees
def view_cells(s: T.State) -> torch.Tensor:
    """Board cell under each view cell: (B, W, W, 2) with W = 2N − 1, centred on the head, heading pointing up."""
    n = s.age.shape[1]
    R = n - 1
    i = torch.arange(2 * R + 1, device=s.age.device)
    ahead = (R - i)[:, None, None]  # (W, 1, 1) rows above the centre are ahead
    right = (i - R)[None, :, None]  # (1, W, 1) columns right of the centre are to the right
    dirs = T.DIRS.to(s.age.device)
    fwd, rgt = dirs[s.dir][:, None, None], dirs[(s.dir + 1) % 4][:, None, None]  # (B, 1, 1, 2)
    return s.head[:, None, None] + ahead * fwd + right * rgt


def planes(s: T.State) -> torch.Tensor:
    """The CNN input (B, 4, W, W): wall (outside the board), body, age/length (1 at the head, 1/length at the
    tail: which cells free up first) and food."""
    n = s.age.shape[1]
    cells = view_cells(s)
    inside = _inside(cells, n)
    age = torch.where(inside, _cell(s.age, cells), 0).float()
    food = (cells == s.food[:, None, None]).all(-1) & inside
    return torch.stack([(~inside).float(), (age > 0).float(), age / s.length[:, None, None], food.float()], 1)


# ---------------------------------------------------------------- the teacher's search (tensor BFS)
def min_propagate(v: torch.Tensor, inc: torch.Tensor, free: torch.Tensor) -> torch.Tensor:
    """Breadth-first search as repeated relaxation, all games and channels at once: on free cells
    v ← min(v, min over the 4 neighbours of v + inc) until nothing changes. v (B, Q, N, N) long, inc (Q,),
    free (B, N, N) bool. inc = 1: distances from the cells where v = 0; inc = 0: the smallest label of each
    connected free region (a flood fill)."""
    inc = inc.view(1, -1, 1, 1)
    blocked = ~free[:, None]
    while True:
        p = F.pad(v, (1, 1, 1, 1), value=FAR)
        nb = torch.minimum(torch.minimum(p[..., :-2, 1:-1], p[..., 2:, 1:-1]),
                           torch.minimum(p[..., 1:-1, :-2], p[..., 1:-1, 2:]))  # fmt: skip
        new = torch.where(blocked, v, torch.minimum(v, nb + inc))
        if torch.equal(new, v):
            return v
        v = new


def distance_field(start: torch.Tensor, free: torch.Tensor) -> torch.Tensor:
    """Steps from the start cells to every cell through free cells: (B, N, N) bool, bool → (B, N, N) long, FAR
    where unreachable."""
    v = torch.where(start, 0, FAR)[:, None]
    return min_propagate(v, torch.ones(1, dtype=torch.long, device=v.device), free)[:, 0]


def search(start: torch.Tensor, free: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Both searches of the teacher in one relaxation: distance from the food and the label of each free region.
    → dist (B, N, N), label (B, N, N) (FAR outside free cells)."""
    B, n, _ = free.shape
    cell = torch.arange(n * n, device=free.device).view(1, n, n)
    v = torch.stack([torch.where(start, 0, FAR), torch.where(free, cell, FAR)], 1)
    v = min_propagate(v, torch.tensor([1, 0], device=free.device), free)
    return v[:, 0], v[:, 1]


def candidates(s: T.State) -> torch.Tensor:
    """Cell the head would enter for each relative action: (B, 3, 2)."""
    d = (s.dir[:, None] + torch.arange(3, device=s.dir.device) - 1) % 4
    return s.head[:, None] + T.DIRS.to(s.head.device)[d]


# ---------------------------------------------------------------- the networks, model axis m first
def cnn_features(x: torch.Tensor, w: dict) -> torch.Tensor:
    """Views x (M, B, 4, W, W), model m sees its own games → h (M, B, F). The model axis becomes the groups of one
    grouped convolution per layer, so the M networks never mix (as in stitch_llm_cnn_dag)."""
    M, B = x.shape[:2]
    z = x.transpose(0, 1).flatten(1, 2)  # (B, M·4, W, W)
    z = F.relu(F.conv2d(z, w["c1"].flatten(0, 1), w["b1"].flatten(), padding=1, groups=M))  # (B, 8M, W, W)
    z = F.relu(F.conv2d(z, w["c2"].flatten(0, 1), w["b2"].flatten(), stride=2, groups=M))  # (B, 16M, S, S)
    z = z.view(B, M, -1).transpose(0, 1)  # (M, B, 16·S·S)
    return F.relu(torch.einsum("mbi,mif->mbf", z, w["fc"]) + w["bf"][:, None])


def router_probs(h: torch.Tensor, router: dict) -> torch.Tensor:
    """p (M, B, K) = softmax(h W^G + b^G): how much each game wants each expert."""
    return torch.softmax(torch.einsum("mbf,mfk->mbk", h, router["W"]) + router["b"][:, None], -1)


def experts_on(U: torch.Tensor, ex: dict) -> torch.Tensor:
    """All experts at once on their own inputs: U (M, K, S, F) → action logits Y (M, K, S, 3)."""
    hid = torch.relu(torch.einsum("mksf,mkfh->mksh", U, ex["W1"]) + ex["b1"][:, :, None])
    return torch.einsum("mksh,mkha->mksa", hid, ex["W2"]) + ex["b2"][:, :, None]


def moe_dense(h: torch.Tensor, g: torch.Tensor, w: dict) -> torch.Tensor:
    """Dense mixture: every expert on every game, z_{mba} = g_{mbk} y_{mkba}.  h (M, B, F), g (M, B, K) → (M, B, 3)."""
    K = g.shape[-1]
    Y = experts_on(h[:, None].expand(-1, K, -1, -1), w["experts"])  # (M, K, B, 3)
    return torch.einsum("mbk,mkba->mba", g, Y)


def moe_top1(h: torch.Tensor, G: torch.Tensor, g: torch.Tensor, w: dict) -> torch.Tensor:
    """Sparse layer: each game runs only its routed expert. G (M, B, K) one-hot, g (M, B, K) gate values (G for a
    fixed router, G ⊙ p for Switch). Dispatch into per-expert buffers, one batched expert call, combine."""
    cap = max(1, int(G.sum(1).max()))  # the largest group: no game is dropped
    D = dispatch(G, cap)  # (M, B, K, S)
    U = torch.einsum("mbks,mbf->mksf", D, h)  # each expert's buffer of games
    return combine(D, g, experts_on(U, w["experts"]))  # (M, B, 3)


def mono_logits(h: torch.Tensor, w: dict) -> torch.Tensor:
    """The monolithic controller, one MLP: h (M, B, F) → (M, B, 3)."""
    hid = torch.relu(torch.einsum("mbf,mfh->mbh", h, w["W1"]) + w["b1"][:, None])
    return torch.einsum("mbh,mha->mba", hid, w["W2"]) + w["b2"][:, None]


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/ai/snake_moe_control/tensors.py
"""Build the concrete tensors: the state of B Snake games, the weights of M controllers stacked on a leading model
axis (one per variant × seed, no nn.Module), and the run settings.

State of game b on an N × N board (rows r, columns c, row 0 at the top):
    age (B, N, N)   0 = empty, else the number of steps until that body cell is free again: the head has
                    age = length, the tail age 1 (so a body cell's age also tells its order along the snake)
    head (B, 2)     row, column of the head          dir (B,)  heading 0 up, 1 right, 2 down, 3 left
    food (B, 2)     row, column of the food          length (B,), alive (B,), hunger (B,) steps since the last food
    eaten (B,)      food eaten so far                steps (B,) moves made
Actions are relative to the heading: 0 turn left, 1 straight, 2 turn right.
"""

from dataclasses import dataclass

import torch

from problems.ai.moe_dag_pipeline.tensors import clone, leaves, n_params, tmap  # noqa: F401  (tree helpers)

DIRS = torch.tensor([[-1, 0], [0, 1], [1, 0], [0, -1]])  # (4, 2) row/column step of up, right, down, left
LEFT, STRAIGHT, RIGHT = 0, 1, 2
ACTIONS = ("left", "straight", "right")
WALL, BODY, AGE, FOOD = 0, 1, 2, 3  # channels of the view
C = 4
START_LENGTH = 3
VARIANTS = ("dense", "switch", "dense_top1", "mono")  # (a) (b) (c) (d)
LABELS = {"random": "random", "dense": "(a) dense", "switch": "(b) Switch top-1", "dense_top1": "(c) dense→top-1",
          "mono": "(d) monolithic", "teacher": "teacher"}  # fmt: skip


@dataclass
class State:
    age: torch.Tensor
    head: torch.Tensor
    dir: torch.Tensor
    food: torch.Tensor
    length: torch.Tensor
    alive: torch.Tensor
    hunger: torch.Tensor
    eaten: torch.Tensor
    steps: torch.Tensor

    def index(self, i) -> "State":
        return State(**{k: v[i] for k, v in vars(self).items()})

    def to(self, dev) -> "State":
        return State(**{k: v.to(dev) for k, v in vars(self).items()})


@dataclass
class Config:
    grid: int = 10  # board N × N
    seeds: int = 2  # models per variant on the model axis
    experts: int = 4  # K
    features: int = 64  # CNN feature width F
    hidden: int = 32  # each expert F → H → 3
    teacher_games: int = 192  # games played by the teacher for the imitation data
    teacher_steps: int = 300  # longest teacher game recorded
    steps: int = 600  # training steps (AdamW), all models at once
    batch: int = 128  # states per model per step
    lr: float = 3e-3
    balance_weight: float = 0.01  # Switch load-balancing coefficient α
    dagger: bool = True  # one DAgger round: label the states the students visit, then train on all
    dagger_games: int = 48
    dagger_steps: int = 200
    eval_games: int = 128  # the same games (same seed) for every policy
    eval_steps: int = 300
    hunger_factor: int = 1  # a game ends after hunger_factor · N² steps without food
    show_games: int = 6  # games in the GIF
    show_steps: int = 160

    def hunger_limit(self) -> int:
        return self.hunger_factor * self.grid * self.grid


# ---------------------------------------------------------------- weights, stacked on the model axis m
def _randn(gen, *shape, scale=1.0):
    return torch.randn(*shape, generator=gen) * scale


def init_cnn(M: int, F: int, grid: int, gen: torch.Generator) -> dict:
    """Perception: conv 3×3 (C → 8) → conv 3×3 stride 2 (8 → 16) → linear → F features, per model m.
    The view is (2N − 1) × (2N − 1); after the stride-2 conv it is S × S with S = N − 1."""
    S = grid - 1
    return {"c1": _randn(gen, M, 8, C, 3, 3, scale=(2 / (9 * C)) ** 0.5), "b1": torch.zeros(M, 8),
            "c2": _randn(gen, M, 16, 8, 3, 3, scale=(2 / 72) ** 0.5), "b2": torch.zeros(M, 16),
            "fc": _randn(gen, M, 16 * S * S, F, scale=(2 / (16 * S * S)) ** 0.5), "bf": torch.zeros(M, F)}  # fmt: skip


def init_moe(M: int, K: int, F: int, H: int, gen: torch.Generator) -> dict:
    """Control: a router F → K and K experts F → H → 3 (stacked on k), per model m."""
    return {
        "router": {"W": _randn(gen, M, F, K, scale=0.1), "b": torch.zeros(M, K)},
        "experts": {"W1": _randn(gen, M, K, F, H, scale=(2 / F) ** 0.5), "b1": torch.zeros(M, K, H),
                    "W2": _randn(gen, M, K, H, 3, scale=(1 / H) ** 0.5), "b2": torch.zeros(M, K, 3)},
    }  # fmt: skip


def init_mono(M: int, F: int, H: int, gen: torch.Generator) -> dict:
    """Control without routing: one MLP F → H → 3 with H chosen to match the MoE's weight count."""
    return {"W1": _randn(gen, M, F, H, scale=(2 / F) ** 0.5), "b1": torch.zeros(M, H),
            "W2": _randn(gen, M, H, 3, scale=(1 / H) ** 0.5), "b2": torch.zeros(M, 3)}  # fmt: skip


def moe_params(cfg: Config) -> int:
    K, F, H = cfg.experts, cfg.features, cfg.hidden
    return F * K + K + K * (F * H + H + H * 3 + 3)


def mono_hidden(cfg: Config) -> int:
    """H of the monolithic MLP so that F·H + H + 3·H + 3 ≈ the MoE controller's weights."""
    return round((moe_params(cfg) - 3) / (cfg.features + 4))


def stack(ws: list[dict]) -> dict:
    """Concatenate models on the model axis."""
    out = {}
    for k, v in ws[0].items():
        out[k] = stack([w[k] for w in ws]) if isinstance(v, dict) else torch.cat([w[k] for w in ws])
    return out


def to(w: dict, dev) -> dict:
    return tmap(w, lambda t: t.to(dev))


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/ai/snake_moe_control/solve.py
"""Solvers: the planning teacher, batched rollouts, imitation learning of all controllers at once (one model axis),
one DAgger round, evaluation on shared games, and the routing analysis.

Training layout on the model axis (S seeds each): [dense MoE × S, Switch MoE × S, monolithic × S]. Evaluation adds
(c): the dense-trained MoE run with top-1 routing, so the evaluated models are [dense, Switch, dense→top-1, mono].
"""

import math
import time

import torch

from problems._common.progress import steps as progress_steps

from . import equation as eq
from . import tensors as T


# ---------------------------------------------------------------- the teacher
def teacher_actions(s: T.State) -> torch.Tensor:
    """A planner: among the three moves, never a deadly one, prefer a move whose flood-fill area can hold the
    snake, then the shortest path to the food (tensor BFS), then the largest area. → actions (B,)."""
    B, n, _ = s.age.shape
    free = s.age <= 1  # the tail cell is free next step
    cand = eq.candidates(s)  # (B, 3, 2)
    inside = eq._inside(cand, n)
    safe = inside & (eq._cell(s.age, cand) <= 1)
    start = torch.zeros_like(free)
    start[torch.arange(B, device=s.age.device), s.food[:, 0], s.food[:, 1]] = True
    dist_map, label = eq.search(start, free)
    dist = eq._cell(dist_map, cand)  # (B, 3) steps from each candidate to the food
    lab = torch.where(safe, eq._cell(label, cand), -1)  # free region each candidate enters
    area = (label.flatten(1)[:, None, :] == lab[..., None]).sum(-1)  # (B, 3) cells in that region
    roomy = area >= s.length[:, None]
    straight = torch.tensor([0.0, 0.5, 0.0], device=s.age.device)
    key = 4e6 * safe + 2e6 * roomy - 1e3 * dist.clamp(max=999) + area + straight
    return key.argmax(1)


def random_actions(B: int, gen: torch.Generator, dev) -> torch.Tensor:
    return torch.randint(3, (B,), generator=gen).to(dev)


# ---------------------------------------------------------------- networks on the model axis
def sub(w: dict, a: int, b: int) -> dict:
    return T.tmap(w, lambda t: t[a:b])


def forward(x: torch.Tensor, P: dict) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Views x (M, B, 4, W, W) → action logits z (M, B, 3), router p and features h of the MoE models.
    P['layout'] = (n_dense, n_top1, n_mono): models in that order on the model axis."""
    nd, nt, nm = P["layout"]
    h = eq.cnn_features(x, P["cnn"])
    hm = h[: nd + nt]
    p = eq.router_probs(hm, P["moe"]["router"])
    zs = []
    if nd:
        zs.append(eq.moe_dense(hm[:nd], p[:nd], sub(P["moe"], 0, nd)))
    if nt:
        G = eq.top1(p[nd:])
        zs.append(eq.moe_top1(hm[nd:], G, G * p[nd:], sub(P["moe"], nd, nd + nt)))
    if nm:
        zs.append(eq.mono_logits(h[nd + nt :], P["mono"]))
    return torch.cat(zs), p, hm


def init_models(cfg: T.Config, dev) -> dict:
    """Seed s gets the same CNN and MoE initialisation in every variant (a paired comparison)."""
    S, K, F, H = cfg.seeds, cfg.experts, cfg.features, cfg.hidden
    cnn = [T.init_cnn(1, F, cfg.grid, torch.Generator().manual_seed(100 + s)) for s in range(S)]
    moe = [T.init_moe(1, K, F, H, torch.Generator().manual_seed(200 + s)) for s in range(S)]
    mono = [T.init_mono(1, F, T.mono_hidden(cfg), torch.Generator().manual_seed(300 + s)) for s in range(S)]
    P = {"cnn": T.stack(cnn * 3), "moe": T.stack(moe * 2), "mono": T.stack(mono)}
    P = {k: T.clone(T.to(v, dev), grad=True) for k, v in P.items()}
    P["layout"] = (S, S, S)
    return P


def eval_models(P: dict, S: int) -> dict:
    """[dense, Switch, dense→top-1 (the dense weights, routed top-1), mono] for evaluation."""
    idx = list(range(S)) + list(range(S, 2 * S)) + list(range(S))
    cnn_idx = idx + list(range(2 * S, 3 * S))
    return {"cnn": T.tmap(P["cnn"], lambda t: t.detach()[cnn_idx]), "moe": T.tmap(P["moe"], lambda t: t.detach()[idx]),
            "mono": T.tmap(P["mono"], lambda t: t.detach()), "layout": (S, 2 * S, S)}  # fmt: skip


# ---------------------------------------------------------------- data: states the teacher labels
POOL_KEYS = ("age", "head", "dir", "food", "length")
EVAL_SEED = 12345


def pool_state(pool: dict, idx: torch.Tensor) -> T.State:
    """Rows idx of a pool of recorded states as a State (only what the view and the teacher need)."""
    z = torch.zeros(len(idx), dtype=torch.long, device=idx.device)
    st = {k: pool[k][idx].long() for k in POOL_KEYS}
    return T.State(**st, alive=torch.ones_like(z, dtype=torch.bool), hunger=z, eaten=z, steps=z)


def add_to_pool(pool: dict, s: T.State, keep: torch.Tensor, action: torch.Tensor | None = None) -> None:
    for k in POOL_KEYS:
        v = getattr(s, k)[keep]
        pool.setdefault(k, []).append(v.to(torch.int16) if k == "age" else v)
    if action is not None:
        pool.setdefault("action", []).append(action[keep])


def cat_pool(pool: dict) -> dict:
    return {k: torch.cat(v) for k, v in pool.items()}


def label(pool: dict, chunk: int = 8192) -> dict:
    """The teacher's action for every recorded state (in chunks: one batched search each)."""
    n = len(pool["dir"])
    idx = torch.arange(n, device=pool["dir"].device)
    pool["action"] = torch.cat([teacher_actions(pool_state(pool, idx[i : i + chunk])) for i in range(0, n, chunk)])
    return pool


def merge(a: dict, b: dict) -> dict:
    return {k: torch.cat([a[k], b[k]]) for k in a}


def cat_states(states: list[T.State]) -> T.State:
    return T.State(**{k: torch.cat([getattr(s, k) for s in states]) for k in vars(states[0])})


def repeat_state(s: T.State, M: int) -> T.State:
    return T.State(**{k: v.repeat(M, *[1] * (v.dim() - 1)) for k, v in vars(s).items()})


def noise(seed: int, B: int, n: int, repeat: int = 1):
    """The food-respawn numbers u (B·repeat, N²) for step t: the same stream for every copy of a game."""
    g = torch.Generator().manual_seed(seed)
    return lambda t: torch.rand(B, n * n, generator=g).repeat(repeat, 1)


def play(cfg: T.Config, s: T.State, act, u_of, n_steps: int, on_step=None) -> T.State:
    """Run games with act(state) → actions until all have ended or n_steps; on_step(t, state, actions)."""
    for t in range(n_steps):
        if not bool(s.alive.any()):
            break
        a = act(s)
        if on_step is not None:
            on_step(t, s, a)
        s = eq.step(s, a, u_of(t).to(s.age.device), cfg.hunger_limit())
    return s


def eval_start(cfg: T.Config) -> T.State:
    return eq.reset(cfg.eval_games, cfg.grid, torch.Generator().manual_seed(EVAL_SEED))


def teacher_run(cfg: T.Config, dev) -> tuple[dict, T.State]:
    """The teacher plays its training games (recorded with its actions, for at most teacher_steps moves) and the
    evaluation games in ONE batch: a search costs about the same for 100 games as for 1000 on a CPU."""
    Bd, n = cfg.teacher_games, cfg.grid
    s = cat_states([eq.reset(Bd, n, torch.Generator().manual_seed(0)), eval_start(cfg)]).to(dev)
    ud, ue = noise(1, Bd, n), noise(EVAL_SEED + 1, cfg.eval_games, n)
    pool: dict = {}

    def record(t, st, a):
        if t < cfg.teacher_steps:
            keep = st.alive.clone()
            keep[Bd:] = False
            add_to_pool(pool, st, keep, a)

    s = play(cfg, s, teacher_actions, lambda t: torch.cat([ud(t), ue(t)]), cfg.eval_steps, record)
    return cat_pool(pool), s.index(slice(Bd, None))


def student_pool(cfg: T.Config, P: dict, dev, seed: int) -> dict:
    """DAgger: the students play greedily; the teacher labels the states they reach."""
    M, B, n = sum(P["layout"]), cfg.dagger_games, cfg.grid
    s = repeat_state(eq.reset(B, n, torch.Generator().manual_seed(seed)), M).to(dev)
    pool: dict = {}

    def act(st):
        with torch.no_grad():
            return forward(eq.planes(st).view(M, B, T.C, 2 * n - 1, 2 * n - 1), P)[0].argmax(-1).flatten()

    play(cfg, s, act, noise(seed + 1, B, n, M), cfg.dagger_steps, lambda t, st, a: add_to_pool(pool, st, st.alive))
    return label(cat_pool(pool))


# ---------------------------------------------------------------- imitation learning
def train(cfg: T.Config, P: dict, pool: dict, n_steps: int, gen: torch.Generator, t0: int = 0, opt=None):
    """Cross-entropy on the teacher's actions (+ α ℒ_aux for the MoE models), every model its own minibatches,
    AdamW (elementwise, so the models stay independent) with warm-up and cosine decay over cfg.steps.
    Returns the loss and accuracy histories (steps, M) and the optimiser."""
    S, W = cfg.seeds, 2 * cfg.grid - 1
    M = 3 * S
    if opt is None:
        opt = torch.optim.AdamW([t for k in ("cnn", "moe", "mono") for t in T.leaves(P[k])], lr=cfg.lr, weight_decay=0)
    n_pool = len(pool["dir"])
    losses, accs = [], []
    for t in progress_steps(n_steps, "imitation"):
        tt = t0 + t
        for g in opt.param_groups:
            g["lr"] = cfg.lr * min(1.0, (tt + 1) / 30) * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * tt / cfg.steps)))
        rows = torch.randint(n_pool, (M, cfg.batch), generator=gen).to(pool["dir"].device)
        x = eq.planes(pool_state(pool, rows.flatten())).view(M, cfg.batch, T.C, W, W)
        y = pool["action"][rows]
        z, p, _ = forward(x, P)
        ce = eq.cross_entropy(z, y)
        aux = eq.load_balance_loss(p, eq.top1(p))  # both MoE variants get the Switch balancing term
        loss = ce + cfg.balance_weight * torch.cat([aux, torch.zeros(S, device=aux.device)])
        opt.zero_grad()
        loss.sum().backward()  # model m's loss reaches only model m's weights
        opt.step()
        losses.append(ce.detach().cpu())
        accs.append((z.argmax(-1) == y).float().mean(1).detach().cpu())
    return torch.stack(losses), torch.stack(accs), opt


# ---------------------------------------------------------------- evaluation and routing
def free_ahead(x: torch.Tensor) -> torch.Tensor:
    """Free cells straight ahead of the head before a wall or body cell, from the views (B, 4, W, W) → (B,)."""
    R = x.shape[-1] // 2
    col = x[:, :, :R, R].flip(-1)  # (B, 4, R) cells ahead, nearest first
    free = (col[:, T.WALL] == 0) & (col[:, T.BODY] == 0)
    return free.long().cumprod(-1).sum(-1)


def evaluate(cfg: T.Config, P: dict, dev) -> dict:
    """The 4S networks and a random policy on the evaluation games (same start, same food stream as the teacher's
    run). Records food eaten, moves survived, the routing of the 3S routed models and the first games for the GIF."""
    S, B, n = cfg.seeds, cfg.eval_games, cfg.grid
    W = 2 * n - 1
    s0 = eval_start(cfg)
    out = {"food": {}, "steps": {}}
    ga = torch.Generator().manual_seed(EVAL_SEED + 2)
    s = play(cfg, s0.to(dev), lambda st: random_actions(len(st.dir), ga, dev), noise(EVAL_SEED + 1, B, n),
             cfg.eval_steps)  # fmt: skip
    out["food"]["random"], out["steps"]["random"] = s.eaten.float().cpu(), s.steps.float().cpu()

    E = eval_models(P, S)
    M, nr = sum(E["layout"]), 3 * S  # 4S models, the first 3S routed
    show = min(cfg.show_games, B)
    rec = {k: [] for k in ("model", "expert", "dist", "ahead", "length", "turn")}
    frames = {k: [] for k in ("age", "food", "alive", "length", "p")}

    def act(st):
        with torch.no_grad():
            x = eq.planes(st)
            z, p, _ = forward(x.view(M, B, T.C, W, W), E)
        a = z.argmax(-1)  # (M, B)
        alive = st.alive.view(M, B)[:nr]
        head, food = st.head.view(M, B, 2)[:nr], st.food.view(M, B, 2)[:nr]
        feats = {"model": torch.arange(nr, device=dev)[:, None].expand(nr, B), "expert": p.argmax(-1),
                 "dist": (head - food).abs().sum(-1), "ahead": free_ahead(x).view(M, B)[:nr],
                 "length": st.length.view(M, B)[:nr], "turn": a[:nr]}  # fmt: skip
        for k, v in feats.items():
            rec[k].append(v[alive].cpu())
        for k in ("age", "food", "alive", "length"):
            v = getattr(st, k)
            frames[k].append(v.view(M, B, *v.shape[1:])[:nr, :show].cpu())
        frames["p"].append(p[:, :show].cpu())
        return a.flatten()

    s = play(cfg, repeat_state(s0, M).to(dev), act, noise(EVAL_SEED + 1, B, n, M), cfg.eval_steps)
    eaten, moves = s.eaten.view(M, B).float().cpu(), s.steps.view(M, B).float().cpu()
    for j, v in enumerate(T.VARIANTS):
        out["food"][v], out["steps"][v] = eaten[j * S : (j + 1) * S], moves[j * S : (j + 1) * S]
    out["routing"] = {k: torch.cat(v) for k, v in rec.items()}
    out["frames"] = {k: torch.stack(v) for k, v in frames.items()}  # (T, nr, show, ...)
    return out


BINS = {"dist": [2, 4, 7], "ahead": [1, 2, 4], "length": [5, 9, 15]}  # bin edges: ≤1, 2–3, 4–6, ≥7 …


def specialisation(routing: dict, nr: int, K: int) -> dict:
    """Per routed model: expert loads, P(expert | feature bin) and the mutual information (bits) between the routed
    expert and binned state features (distance to food, free cells ahead, body length) and the move the controller
    makes. 0 bits = the routing tells nothing about that feature; log2 K bits is the most it can."""
    feats = (*BINS, "turn")
    out = {"load": torch.zeros(nr, K), "mi": {k: torch.zeros(nr) for k in feats},
           "table": {k: torch.zeros(nr, 3 if k == "turn" else 4, K) for k in feats}}  # fmt: skip
    for m in range(nr):
        sel = routing["model"] == m
        e = routing["expert"][sel]
        if len(e) == 0:
            continue
        out["load"][m] = torch.bincount(e, minlength=K).float() / len(e)
        for k in feats:
            v = routing[k][sel]
            b = v if k == "turn" else torch.bucketize(v, torch.tensor(BINS[k]), right=True)
            joint = torch.zeros(out["table"][k].shape[1], K)
            joint.index_put_((b, e), torch.ones(len(e)), accumulate=True)
            out["table"][k][m] = joint / joint.sum(1, keepdim=True).clamp(min=1)  # P(expert | bin)
            pj = joint / joint.sum()
            indep = pj.sum(1, keepdim=True) * pj.sum(0, keepdim=True)
            nz = pj > 0
            out["mi"][k][m] = (pj[nz] * torch.log2(pj[nz] / indep[nz])).sum()
    return out


# ---------------------------------------------------------------- everything
def run_all(cfg: T.Config, dev) -> dict:
    tm = {}
    t0 = time.time()
    pool, s_teacher = teacher_run(cfg, dev)
    tm["teacher (data + eval)"] = time.time() - t0
    P = init_models(cfg, dev)
    gen = torch.Generator().manual_seed(7)
    first = int(cfg.steps * 0.6) if cfg.dagger else cfg.steps
    t1 = time.time()
    loss, acc, opt = train(cfg, P, pool, first, gen)
    n_teacher = len(pool["dir"])
    if cfg.dagger:
        t2 = time.time()
        pool = merge(pool, student_pool(cfg, P, dev, seed=1))
        tm["DAgger data"] = time.time() - t2
        l2, a2, _ = train(cfg, P, pool, cfg.steps - first, gen, first, opt)
        loss, acc = torch.cat([loss, l2]), torch.cat([acc, a2])
    tm["training"] = time.time() - t1 - tm.get("DAgger data", 0.0)
    t3 = time.time()
    ev = evaluate(cfg, P, dev)
    tm["evaluation"] = time.time() - t3
    ev["food"]["teacher"], ev["steps"]["teacher"] = s_teacher.eaten.float().cpu(), s_teacher.steps.float().cpu()
    return {"cfg": cfg, "loss": loss, "acc": acc, "dagger_at": first if cfg.dagger else None,
            "pool": (n_teacher, len(pool["dir"])), **ev, "spec": specialisation(ev["routing"], 3 * cfg.seeds,
            cfg.experts), "time": tm, "params": {k: T.n_params(P[k]) for k in ("cnn", "moe", "mono")}}  # fmt: skip


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/ai/snake_moe_control/plot.py
"""Animate the result: six games played at once by a routed controller (each board's frame in the colour of the
expert the router chose at that step), the router weights of every game, the router of game 1 over time, the
imitation learning curves and the score of every policy with one dot per seed. Also a static figure of what the
experts are used for (routing vs state features) and the CSV tables."""

import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec

from problems._common.plotting import ACCENT, frame_indices, save_png
from problems.ai.chain_of_thought.plot import save_gif_shared_palette

from . import tensors as T

EXPERT_COLORS = ["#ea580c", "#16a34a", "#9333ea", "#0891b2", "#db2777", "#65a30d", "#a16207", "#4b5563"]
POLICY_COLORS = {"random": "#9ca3af", "dense": ACCENT[0], "switch": ACCENT[1], "dense_top1": ACCENT[3],
                 "mono": ACCENT[2], "teacher": "#374151"}  # fmt: skip
POLICIES = ("random", "dense", "switch", "dense_top1", "mono", "teacher")
SHORT = {"random": "random", "dense": "(a)\ndense", "switch": "(b)\nSwitch", "dense_top1": "(c)\ndense→top-1",
         "mono": "(d)\nmono", "teacher": "teacher"}  # fmt: skip
FEATURES = {"dist": ("distance to food (cells)", ["≤1", "2–3", "4–6", "≥7"]),
            "ahead": ("free cells ahead (cells)", ["0", "1", "2–3", "≥4"]),
            "length": ("body length (cells)", ["≤4", "5–8", "9–14", "≥15"]),
            "turn": ("controller's move (dimensionless)", ["left", "straight", "right"])}  # fmt: skip


def shown_variant(res: dict) -> str:
    """(c) plays in the GIF: every game runs exactly one expert, and its router uses all of them (the Switch
    router of (b) tends to collapse onto one or two experts; routing.png shows all three routed variants)."""
    return "dense_top1"


def board_rgb(age: np.ndarray, length: int, food: np.ndarray) -> np.ndarray:
    """N × N board → RGB: light grey cells, the body from dark (head) to light blue (tail), the food red."""
    n = age.shape[0]
    img = np.ones((n, n, 3)) * np.array([0.95, 0.96, 0.97])
    frac = age / max(length, 1)
    dark, light = np.array([0.09, 0.25, 0.69]), np.array([0.66, 0.80, 0.98])
    body = age > 0
    img[body] = light + (dark - light) * frac[body][:, None]
    img[food[0], food[1]] = [0.86, 0.15, 0.15]
    return img


def animate(res: dict, path: Path, n_frames: int = 80) -> Path:
    cfg, S, K = res["cfg"], res["cfg"].seeds, res["cfg"].experts
    v = shown_variant(res)
    mi = T.VARIANTS.index(v) * S  # seed 1 of that variant
    fr = res["frames"]
    n_t = min(cfg.show_steps, fr["age"].shape[0])
    show = fr["age"].shape[2]
    frames = frame_indices(n_t, n_frames)
    age, food, alive = fr["age"][:, mi].numpy(), fr["food"][:, mi].numpy(), fr["alive"][:, mi].numpy()
    length, p = fr["length"][:, mi].numpy(), fr["p"][:, mi].numpy()  # (T, show), (T, show, K)

    fig = plt.figure(figsize=(14.5, 9.4))
    gs = GridSpec(3, 6, figure=fig, height_ratios=[1, 1, 1.05], hspace=0.55, wspace=0.6,
                  left=0.05, right=0.985, top=0.93, bottom=0.08)  # fmt: skip
    boards = [fig.add_subplot(gs[i // 3, i % 3]) for i in range(min(show, 6))]
    ax_bar, ax_time = fig.add_subplot(gs[0, 3:]), fig.add_subplot(gs[1, 3:])
    ax_learn, ax_score = fig.add_subplot(gs[2, :3]), fig.add_subplot(gs[2, 3:])
    fig.suptitle(f"Snake, 6 of {cfg.eval_games} games batched, controller {T.LABELS[v]} (seed 1): "
                 "frame colour = routed expert", fontsize=13)  # fmt: skip

    ims = []
    for g, ax in enumerate(boards):
        ims.append(ax.imshow(board_rgb(age[0, g], length[0, g], food[0, g]), interpolation="nearest"))
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        if g >= 3:
            ax.set_xlabel("column (cells)", fontsize=10)
        if g % 3 == 0:
            ax.set_ylabel("row (cells)", fontsize=10)

    # router weights of the six games now
    lefts = np.zeros(len(boards))
    bars = []
    for k in range(K):
        bars.append(ax_bar.barh(np.arange(len(boards)), p[0, : len(boards), k], left=lefts, color=EXPERT_COLORS[k],
                                label=f"expert {k + 1}", height=0.7))  # fmt: skip
    ax_bar.set_xlim(0, 1)
    ax_bar.set_ylim(len(boards) - 0.5, -0.5)
    ax_bar.set_yticks(np.arange(len(boards)), [f"game {g + 1}" for g in range(len(boards))])
    ax_bar.set_xlabel("router weight $p_k$ (dimensionless)")
    ax_bar.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=K, fontsize=9, frameon=False)

    # router of game 1 over time
    t_axis = np.arange(n_t)
    ax_time.set_xlim(0, n_t - 1)
    ax_time.set_ylim(0, 1)
    ax_time.set_xlabel("game step $t$ (moves)")
    ax_time.set_ylabel("$p_k$ (dimensionless)")
    ax_time.set_title("router of game 1 over time (stacked $p_k$)")
    cursor = ax_time.axvline(0, color="#111827", lw=1.2)

    # learning curves (static)
    loss = res["loss"].numpy()
    steps = np.arange(1, len(loss) + 1)
    for j, name in enumerate(("dense", "switch", "mono")):
        c = POLICY_COLORS[name]
        for s in range(S):
            ax_learn.plot(steps, _smooth(loss[:, j * S + s]), color=c, lw=1.0, alpha=0.6,
                          label=T.LABELS[name] if s == 0 else None)  # fmt: skip
    if res["dagger_at"]:
        ax_learn.axvline(res["dagger_at"], color="#9ca3af", ls="--", lw=1)
        ax_learn.text(res["dagger_at"], 0.97, "DAgger data added ", transform=ax_learn.get_xaxis_transform(),
                      va="top", ha="right", fontsize=9, color="#4b5563")  # fmt: skip
    ax_learn.set_yscale("log")
    ax_learn.set_xlim(1, len(loss))
    lo, hi = np.nanmin(_smooth(loss[:, 0])), np.nanmax(loss)
    ax_learn.set_ylim(min(lo, np.nanmin(loss[len(loss) // 2 :])) * 0.8, hi * 1.2)
    ax_learn.set_xlabel("training step (minibatches)")
    ax_learn.set_ylabel("imitation loss (nats)")
    ax_learn.set_title("learning: cross-entropy vs the teacher (one line per seed)")
    ax_learn.legend(fontsize=9, loc="upper right")

    # scores (static)
    _score_bars(ax_score, res)

    state = {"fill": []}

    def update(f):
        for g, ax in enumerate(boards):
            ims[g].set_data(board_rgb(age[f, g], length[f, g], food[f, g]))
            k = int(p[f, g].argmax())
            col = EXPERT_COLORS[k] if alive[f, g] else "#9ca3af"
            for sp in ax.spines.values():
                sp.set_edgecolor(col)
                sp.set_linewidth(5)
            ax.set_title(f"game {g + 1}: length {length[f, g]}" + ("" if alive[f, g] else " (ended)"),
                         fontsize=11, color="#111827" if alive[f, g] else "#6b7280")  # fmt: skip
        lefts = np.zeros(len(boards))
        for k in range(K):
            for g, rect in enumerate(bars[k]):
                w = p[f, g, k] if alive[f, g] else 0.0
                rect.set_x(lefts[g])
                rect.set_width(w)
                lefts[g] += w
        ax_bar.set_title(f"router weights now, step $t$ = {f}")
        for c in state["fill"]:
            c.remove()
        upto = int(f) + 1
        live = alive[:upto, 0]
        pp = np.where(live[:, None], p[:upto, 0], np.nan)
        state["fill"] = list(ax_time.stackplot(t_axis[:upto], pp.T, colors=EXPERT_COLORS[:K], alpha=0.9))
        cursor.set_xdata([f, f])

    return save_gif_shared_palette(fig, update, frames, path, fps=6)


def _smooth(y: np.ndarray, k: int = 25) -> np.ndarray:
    if len(y) < k:
        return y
    c = np.cumsum(np.insert(y, 0, 0.0))
    out = (c[k:] - c[:-k]) / k
    return np.concatenate([np.full(k - 1, np.nan), out])


def _score_bars(ax, res: dict) -> None:
    hi = max(float(res["food"][v].mean(-1).max()) for v in POLICIES)
    for i, v in enumerate(POLICIES):
        x = res["food"][v]
        per_seed = x.mean(-1).numpy() if x.dim() > 1 else np.array([float(x.mean())])
        ax.bar(i, per_seed.mean(), color=POLICY_COLORS[v], alpha=0.85, width=0.7)
        ax.scatter(np.full(len(per_seed), i) + np.linspace(-0.12, 0.12, len(per_seed)), per_seed, color="#111827",
                   s=18, zorder=3, label="one seed" if i == 1 else None)  # fmt: skip
        ax.text(i, per_seed.mean() + 0.02 * hi, f"{per_seed.mean():.1f}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(range(len(POLICIES)), [SHORT[v] for v in POLICIES], fontsize=10)
    ax.set_ylim(0, hi * 1.18 + 0.5)
    ax.set_xlim(-0.6, len(POLICIES) - 0.4)
    ax.set_ylabel("food eaten per game (count)")
    cfg = res["cfg"]
    ax.set_title(f"score: {cfg.eval_games} shared games, ≤ {cfg.eval_steps} moves")
    ax.legend(fontsize=9, loc="upper left")


def routing_png(res: dict, path: Path) -> Path:
    """P(expert | state feature bin) for each routed variant (seed 1), with the mutual information of every seed."""
    S, K = res["cfg"].seeds, res["cfg"].experts
    sp = res["spec"]
    fig, axes = plt.subplots(3, 4, figsize=(15, 10), squeeze=False)
    for r, v in enumerate(("dense", "switch", "dense_top1")):
        m = r * S
        for c, (feat, (xlabel, ticks)) in enumerate(FEATURES.items()):
            ax = axes[r, c]
            tab = sp["table"][feat][m].numpy()  # (bins, K)
            bottom = np.zeros(len(tab))
            for k in range(K):
                ax.bar(np.arange(len(tab)), tab[:, k], bottom=bottom, color=EXPERT_COLORS[k], width=0.75,
                       label=f"expert {k + 1}")  # fmt: skip
                bottom += tab[:, k]
            ax.set_xticks(range(len(ticks)), ticks)
            ax.set_ylim(0, 1)
            ax.set_xlabel(xlabel)
            mis = ", ".join(f"{float(sp['mi'][feat][r * S + s]):.2f}" for s in range(S))
            ax.set_title(f"{T.LABELS[v]}: MI {mis} bit", fontsize=11)
            if c == 0:
                ax.set_ylabel("share routed (dimensionless)")
    axes[0, 3].legend(fontsize=9, loc="upper right", bbox_to_anchor=(1.0, 1.0))
    fig.suptitle("What the router uses: share of states sent to each expert per feature bin (seed 1); "
                 "MI = mutual information, one value per seed", fontsize=13)  # fmt: skip
    fig.tight_layout()
    return save_png(fig, path)


def write_tables(res: dict, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    S = res["cfg"].seeds
    with (out_dir / "scores.csv").open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["policy", "seed", "games", "food_mean", "food_std", "moves_mean"])
        for v in POLICIES:
            x, st = res["food"][v], res["steps"][v]
            rows = [(0, x, st)] if x.dim() == 1 else [(s + 1, x[s], st[s]) for s in range(S)]
            for s, xs, ss in rows:
                w.writerow([v, s or "-", len(xs), f"{float(xs.mean()):.3f}", f"{float(xs.std()):.3f}",
                            f"{float(ss.mean()):.1f}"])  # fmt: skip
    sp = res["spec"]
    with (out_dir / "routing.csv").open("w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["variant", "seed", "feature", "bin", "expert", "share_routed", "mutual_information_bit"])
        for r, v in enumerate(("dense", "switch", "dense_top1")):
            for s in range(S):
                m = r * S + s
                for k in range(res["cfg"].experts):
                    w.writerow([v, s + 1, "all", "all", k + 1, f"{float(sp['load'][m, k]):.4f}", ""])
                for feat, (_, ticks) in FEATURES.items():
                    for b, tick in enumerate(ticks):
                        for k in range(res["cfg"].experts):
                            w.writerow([v, s + 1, feat, tick, k + 1, f"{float(sp['table'][feat][m, b, k]):.4f}",
                                        f"{float(sp['mi'][feat][m]):.4f}"])  # fmt: skip


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'grid': 10,
    'seeds': 2,
    'experts': 4,
    'steps': 600,
    'balance_weight': 0.01,
    'dagger': 1,
    'eval_games': 128
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.ai.snake_moe_control")]:
    del sys.modules[name]
main = importlib.import_module("problems.ai.snake_moe_control.main")
problem_dir = Path("problems/ai/snake_moe_control")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
